In [192]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [193]:
df = pd.read_csv('./data/spam_preprocessed.csv', encoding='latin-1')
# vectorizer = TfidfVectorizer(stop_words='english', max_features=3000, ngram_range=(1, 2))
# vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=10000, min_df=2)

In [194]:
full_vectorizer = TfidfVectorizer(stop_words='english')
full_vectorizer.fit(df['text'])
print(len(full_vectorizer.vocabulary_))

7868


In [195]:
df['vectorized_message'] = vectorizer.fit_transform(df['text']).toarray().tolist()
df['sum_vector'] = df['vectorized_message'].apply(lambda x: sum(x) if isinstance(x, list) else 0)

# for m in range(len(df['text'])):
#     try:
#         print(f"Index: {m}, Message: {df['text'].iloc[m]}")
#         vec = vectorizer.fit_transform([df['text'].iloc[m]]).toarray()[0]
#         df.at[m, 'vectorized_message'] = vec.tolist()
#     except Exception as e:
#         print(f"Index: {m}, Message: {df['text'].iloc[m]}")
#         print(f"Error vectorizing message: {e}")

# df = df[df['vectorized_message'].notnull()]
print(df.shape)
df[df['sum_vector'] > 0]

(5570, 4)


,label,text,vectorized_message,sum_vector
0,0,go until jurong point crazy available only in ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3.254600
1,0,ok lar joking wif u oni,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2.396007
2,1,free entry in 2 a wkly comp to win fa cup fina...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",5.860550
3,0,u dun say so early hor u c already then say,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2.185426
4,0,nah i don t think he goes to usf he lives arou...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2.600740
...,...,...,...,...
5565,1,this is the 2nd time we have tried 2 contact u...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",5.323360
5566,0,will b going to esplanade fr home,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.937745
5567,0,pity was in mood for that so any other suggest...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.000000
5568,0,the guy did some bitching but i acted like i d...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3.064634


In [196]:
x = np.vstack(df['vectorized_message'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [197]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
classifier = MultinomialNB()
model = classifier.fit(X_train, y_train)
y_pred = model.predict(X_test)

X_train shape: (4456, 8640)
X_test shape: (1114, 8640)


In [198]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9776
Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       947
           1       1.00      0.85      0.92       167

    accuracy                           0.98      1114
   macro avg       0.99      0.93      0.95      1114
weighted avg       0.98      0.98      0.98      1114

Confusion Matrix:
[[947   0]
 [ 25 142]]


In [199]:
test_spam_data = pd.read_json('./data/test_spam_data.json')
test_spam_data.replace({'label': {'ham': 0, 'spam': 1}}, inplace=True)
test_spam_data['vectorized_message'] = vectorizer.transform(test_spam_data['email_text']).toarray().tolist()
test_spam_data = test_spam_data.sample(frac=1, random_state=1290).reset_index(drop=True)
test_spam_data['label'] = test_spam_data['label'].astype(int)

In [200]:
y_test_labels = test_spam_data['label']
separate_X_test = np.vstack(test_spam_data['vectorized_message'])
random_y_pred = model.predict(separate_X_test)
confidence = model.predict_proba(separate_X_test)

In [201]:
test_accuracy = accuracy_score(y_test_labels, random_y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")
print("Test Classification Report:")
print(classification_report(y_test_labels, random_y_pred))
print("Test Confusion Matrix:")
print(confusion_matrix(y_test_labels, random_y_pred))

Test Accuracy: 0.5294
Test Classification Report:
              precision    recall  f1-score   support

           0       0.38      1.00      0.56         5
           1       1.00      0.33      0.50        12

    accuracy                           0.53        17
   macro avg       0.69      0.67      0.53        17
weighted avg       0.82      0.53      0.52        17

Test Confusion Matrix:
[[5 0]
 [8 4]]


In [202]:
test_spam_data['predicted'] = random_y_pred
test_spam_data['confidence_spam'] = confidence[:, 1] * 100
test_spam_data[['label', 'predicted', 'confidence_spam', 'email_text']]

,label,predicted,confidence_spam,email_text
0,1,1,81.791163,Congratulations! You have won a $1000 Amazon g...
1,0,0,0.937616,"Hi John, attached is the updated report you re..."
2,0,0,2.047662,Here are the notes from today's client meeting...
3,1,1,62.614935,"Winner, off, Congratulations, free"
4,0,0,2.666633,Meeting reminder: The project review is schedu...
5,1,0,14.891078,Act now! Lowest mortgage rates available. Pre-...
6,1,0,28.146852,Urgent! Your bank account has been suspended. ...
7,0,0,15.809763,Thank you for your purchase. Your order will b...
8,1,0,7.159043,Exclusive deal just for you. Get 90% off luxur...
9,1,0,39.954318,"Dear customer, your package delivery failed. C..."
